# GO-term covariate attention

In [ ]:
mode = "subset"


In [ ]:
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import LinearSegmentedColormap

colors = [
    "#70acc0",
    "#c12075",
    "#003754",
    "#4c626c",
    "#be9e7c",
    "#723c5d",
    "#9d9c9c",
    "#00695c",
    "#e57373",
]
colors_name = [
    "graublau",
    "purpur",
    "blau",
    "dunkelgrau",
    "bronze",
    "violett",
    "hellgrau",
    "gedämpftes teal",
    "sanftes koralle",
]
sns.set_palette(sns.color_palette(colors))
base_color = "#70acc0"
light_palette = sns.light_palette(base_color, as_cmap=False)
colors = light_palette + ["#4c626c", "#003754"]
cmap = LinearSegmentedColormap.from_list("custom_cmap", colors)
colors_heat = ["#c12075", "#f0f0f0", "#003754"]
cmap_heat = LinearSegmentedColormap.from_list("custom_diverging", colors_heat, N=256)
colors_heat2 = ["#be9e7c", "#f0f0f0", "#4c626c"]
cmap_heat2 = LinearSegmentedColormap.from_list("custom_diverging2", colors_heat2, N=256)
mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42
mpl.rcParams["svg.fonttype"] = "none"

import re as _re
import matplotlib.figure as _mpl_figure

def _fix_svg_font_declarations(_svg_path):
    """Convert matplotlib's unsupported CSS 'font:' shorthand (labeled px, but
    actually point-sized, in a pt-calibrated viewBox) into explicit
    font-family/font-size/font-weight/font-style, which is what Figma's SVG
    importer requires to resolve size and style correctly."""
    try:
        with open(_svg_path, "r", encoding="utf-8") as _f:
            _content = _f.read()
    except OSError:
        return
    def _convert(_m):
        _oblique = _m.group(1)
        _weight = _m.group(2)
        _pt = float(_m.group(3)) * 0.75
        _family = _m.group(4)
        _font_weight = "bold" if _weight else "normal"
        _font_style = "oblique" if _oblique else "normal"
        return (
            f"font-family: {_family}; font-size: {_pt:g}pt; "
            f"font-weight: {_font_weight}; font-style: {_font_style}"
        )
    _pattern = _re.compile(r'font:\s*(oblique\s+)?(?:(\d+)\s+)?([0-9.]+)px\s+([^;"]+)')
    _new_content = _pattern.sub(_convert, _content)
    if _new_content != _content:
        with open(_svg_path, "w", encoding="utf-8") as _f:
            _f.write(_new_content)

_orig_figure_savefig = _mpl_figure.Figure.savefig

def _rasterize_dense_elements(_fig):
    for _ax in _fig.get_axes():
        for _coll in list(_ax.collections):
            try:
                _coll.set_rasterized(True)
            except Exception:
                pass
        for _im in _ax.get_images():
            try:
                _im.set_rasterized(True)
            except Exception:
                pass

def _patched_figure_savefig(self, fname, *args, **kwargs):
    if isinstance(fname, str) and fname.lower().endswith(".svg"):
        _rasterize_dense_elements(self)
        kwargs.setdefault("dpi", 300)
    _result = _orig_figure_savefig(self, fname, *args, **kwargs)
    if isinstance(fname, str) and fname.lower().endswith(".svg"):
        _fix_svg_font_declarations(fname)
    return _result

_mpl_figure.Figure.savefig = _patched_figure_savefig
mpl.rcParams["figure.dpi"] = 500.0
mpl.rcParams["savefig.transparent"] = True
mpl.rcParams["savefig.format"] = "png"

mpl.rcParams["font.family"] = "sans-serif"
mpl.rcParams["font.sans-serif"] = ["DejaVu Sans"]

mpl.rcParams["axes.edgecolor"] = "black"
mpl.rcParams["axes.linewidth"] = 0.8
mpl.rcParams["axes.grid"] = True
mpl.rcParams["grid.color"] = "grey"
mpl.rcParams["grid.linestyle"] = "--"
mpl.rcParams["grid.linewidth"] = 0.5

mpl.rcParams["legend.frameon"] = False
mpl.rcParams["legend.loc"] = "best"

mpl.rcParams["xtick.major.size"] = 3
mpl.rcParams["xtick.minor.size"] = 2
mpl.rcParams["ytick.major.size"] = 3
mpl.rcParams["ytick.minor.size"] = 2
mpl.rcParams["xtick.major.width"] = 0.4
mpl.rcParams["xtick.minor.width"] = 0.3
mpl.rcParams["ytick.major.width"] = 0.4
mpl.rcParams["ytick.minor.width"] = 0.3
scaling_factor = 1
if mode == "all":
    base_font_size = 4
else:
    base_font_size = 6
base_axes_title_size = base_font_size + 1
base_axes_label_size = base_font_size
base_tick_label_size = base_font_size - 0.5
base_legend_font_size = base_font_size - 0.5
base_figure_title_size = base_font_size + 2
mpl.rcParams["font.size"] = base_font_size * scaling_factor
mpl.rcParams["axes.titlesize"] = base_axes_title_size * scaling_factor
mpl.rcParams["axes.labelsize"] = base_axes_label_size * scaling_factor
mpl.rcParams["xtick.labelsize"] = base_tick_label_size * scaling_factor
mpl.rcParams["ytick.labelsize"] = base_tick_label_size * scaling_factor
mpl.rcParams["legend.fontsize"] = base_legend_font_size * scaling_factor
mpl.rcParams["figure.titlesize"] = base_figure_title_size * scaling_factor

mpl.rcParams["patch.linewidth"] = 0.7

mpl.rcParams["figure.autolayout"] = True


In [ ]:
def set_figure_params(column, height):

    single_column_width = 3.5
    double_column_width = 7.0

    if column == "single":
        figure_width = single_column_width
    elif column == "double":
        figure_width = double_column_width
    else:
        figure_width = column
    scaling_factor = figure_width / single_column_width
    scaling_factor = 1

    mpl.rcParams["figure.figsize"] = (figure_width, height)


In [ ]:
import sys
import os

_OUTDIR_ABS = os.path.join(os.getcwd(), "outputs", "fig5b_S13b_S24_neurips2021_cite_bmmc_covariate_attention_analysis")
os.makedirs(_OUTDIR_ABS, exist_ok=True)

sys.path.append(os.path.abspath("../.experiment_data/primary/benchmarking"))

import contextlib
import functools

_POSTPROC_CWD = os.getcwd()

@contextlib.contextmanager
def _in_primary_dir():
    os.chdir(os.path.join(_POSTPROC_CWD, "../.experiment_data/primary"))
    try:
        yield
    finally:
        os.chdir(_POSTPROC_CWD)

with _in_primary_dir():
    import muon as _muon
    _orig_sample_obs = _muon.pp.sample_obs
    def _patched_sample_obs(data, *a, **kw):
        return _orig_sample_obs(data, *a, **kw).copy()
    _muon.pp.sample_obs = _patched_sample_obs
    import datasets

for _name in dir(datasets):
    if _name.startswith("load_"):
        def _make_wrapper(fn):
            @functools.wraps(fn)
            def _wrapper(*a, **kw):
                with _in_primary_dir():
                    return fn(*a, **kw)
            return _wrapper
        setattr(datasets, _name, _make_wrapper(getattr(datasets, _name)))
import glob
import pandas as pd
import scipy
import numpy as np
import os
from tqdm import tqdm
import pickle
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import spearmanr, pearsonr
from scipy.spatial.distance import euclidean
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pickle
import glob
from collections import Counter
from pathlib import Path
from goatools.obo_parser import GODag
from goatools.anno.gaf_reader import GafReader
import glob
import pandas as pd
import scipy
import numpy as np
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import ListedColormap
import textwrap
from matplotlib.colors import TwoSlopeNorm


In [ ]:
obodag = GODag(
    os.path.join(
        os.path.abspath("../.experiment_data/primary"), "preprocessing", "resources", "ebi_ensembl_go_mappings", "go-basic.obo"
    )
)


In [ ]:
import pandas as pd

ensembl_gene_id_external_gene_name = pd.read_csv(
    os.path.abspath(
        os.path.join(
            "..", "preprocessing", "resources", "ensembl_gene_id_gene_name_mapping",
            "ensembl_gene_id_gene_name_mapping.csv",
        )
    )
)
ensembl_gene_id_external_gene_name = ensembl_gene_id_external_gene_name.rename(
    columns={"Gene stable ID": "gene_stable_id"}
)
ensembl_gene_id_external_gene_name = ensembl_gene_id_external_gene_name[
    ~ensembl_gene_id_external_gene_name["Gene name"].isnull()
]


In [ ]:
(
    mdata,
    mdata_counts,
    adata,
    train_indices,
    val_indices,
    n_genes,
    n_regions,
    n_snps,
    labels_key,
    further_labels_keys,
    batch_key,
    patient_key,
    further_batch_keys,
    further_continuous_batch_keys,
    n_patient_covariates,
    protein_expression_obsm_key,
) = datasets.load_neurips2021_cite_BMMC(1.0, False, True)


In [ ]:
import pickle

with open(
    "../.experiment_data/primary/_biological_process_go-basic_goa_human_ensembl_gene_mapping_2024-08-08_goa_human_isoform_ensembl_gene_mapping_2024-08-08_goa_human_rna_ensembl_gene_mapping_2024-08-08.v2_processed.pkl",
    "rb",
) as f:
    ensembl2go = pickle.load(f)


In [ ]:
go_terms_of_interest = ["GO:0002682"]
genes_of_interest = []
for go_term in go_terms_of_interest:
    print(go_term)
    for gene in ensembl2go.keys():
        if go_term in ensembl2go[gene]:
            print(gene)
            genes_of_interest.append(gene)
    print("%%%")


In [ ]:
result_go0002682 = []
from collections import defaultdict

ensembl2go_restrict_go0002682 = defaultdict(set)
for go in tqdm(obodag.keys()):
    if "GO:0002682" in obodag[go].get_all_parents():
        for gene in ensembl2go.keys():
            if go in ensembl2go[gene]:
                result_go0002682.append(gene)
                ensembl2go_restrict_go0002682[gene].add(go)
result_go0002682


In [ ]:
genes_go0002682 = [
    mdata_counts.mod["rna"].var[mdata_counts.mod["rna"].var["gene_stable_id"] == gene].index[0]
    for gene in set(genes_of_interest) & set(mdata_counts.mod["rna"].var["gene_stable_id"])
]


In [ ]:
phenotypes_labels_dotplots = [
    "CD8+ T CD57+ CD45RA+",
    "CD4+ T CD314+ CD45RA+",
    "CD8+ T TIGIT+ CD45RO+",
    "CD8+ T TIGIT+ CD45RA+",
    "CD8+ T CD57+ CD45RO+",
    "NK",
    "CD4+ T activated integrinB7+",
    "CD4+ T naive",
    "CD4+ T activated",
    "CD8+ T naive",
    "CD16+ Mono",
    "CD14+ Mono",
    "cDC2",
    "G/M prog",
    "cDC1",
    "Reticulocyte",
    "Normoblast",
    "Plasmablast IGKC-",
    "Plasmablast IGKC+",
    "Plasma cell IGKC-",
    "Plasma cell IGKC+",
    "Naive CD20+ B IGKC-",
    "Naive CD20+ B IGKC+",
    "Erythroblast",
    "MK/E prog",
    "Lymph prog",
]


In [ ]:
go_graph_csv = pd.read_pickle(
    "../.experiment_data/primary/experiments_new/sparsevi_neurips2021_cite_bmmc_1.0_scib_hvar_True_batscib_True_patscib_False_mmixTrue_no_gating_scarches_False_ec_True_dic_True_ddic_True_batcov_True_patcov_False_modtype_go_genelayerint_ENCFF173VDJ_gaf_standard_with_rna_s0/default/calc_go_terms_cell_type/expression_cell_type_vs_rest_Plasma_cell_IGKC+_res_unp_act_mask_False_flip_False_go_terms_evaluated_graph_csv.pkl"
)
go_graph_csv


In [ ]:
for go_term in ensembl2go_restrict_go0002682["ENSG00000116824"]:
    if go_term in go_graph_csv.index:
        print(go_term)
        print(obodag[go_term].name)


In [ ]:
for go_term in ensembl2go_restrict_go0002682["ENSG00000271503"]:
    if go_term in go_graph_csv.index:
        print(go_term)
        print(obodag[go_term].name)


In [ ]:
for go_term in ensembl2go_restrict_go0002682["ENSG00000077984"]:
    if go_term in go_graph_csv.index:
        print(go_term)
        print(obodag[go_term].name)


In [ ]:
for go_term in ensembl2go_restrict_go0002682["ENSG00000277943"]:
    if go_term in go_graph_csv.index:
        print(go_term)
        print(obodag[go_term].name)


In [ ]:
for go_term in ensembl2go_restrict_go0002682["ENSG00000111796"]:
    if go_term in go_graph_csv.index:
        print(go_term)
        print(obodag[go_term].name)


In [ ]:
for go_term in ensembl2go_restrict_go0002682["ENSG00000134539"]:
    if go_term in go_graph_csv.index:
        print("In Graph")
    print(go_term)
    print(obodag[go_term].name)


In [ ]:
path = "../.experiment_data/primary/experiments_new/sparsevi_neurips2021_cite_bmmc_1.0_scib_hvar_True_batscib_True_patscib_False_mmixTrue_no_gating_scarches_False_ec_True_dic_True_ddic_True_batcov_True_patcov_False_fbatcov_fcbatcov_modtype_go_genelayerint_ENCFF173VDJ_gaf_standard_with_rna_s0/default/covariate_attention_cell_type/"


In [ ]:
path_alt = "../.experiment_data/primary/experiments_new/sparsevi_neurips2021_cite_bmmc_1.0_scib_hvar_True_batscib_True_patscib_False_mmixTrue_no_gating_scarches_False_ec_True_dic_True_ddic_True_batcov_True_patcov_False_fbatcov_fcbatcov_modtype_go_genelayerint_ENCFF173VDJ_gaf_standard_with_rna_s0/default/reload_interpretability/covariate_attention_cell_type/"


In [ ]:
path_shuff = "../.experiment_data/primary/experiments_new/sparsevi_neurips2021_cite_bmmc_1.0_scib_hvar_True_batscib_True_patscib_False_mmixTrue_no_gating_scarches_False_ec_True_dic_True_ddic_True_batcov_True_patcov_False_fbatcov_fcbatcov_modtype_go_genelayerint_ENCFF173VDJ_gaf_standard_with_rna_gshuff_s0/default/covariate_attention_cell_type/"


In [ ]:
modality_categorical_covariate_keys = np.load(
    os.path.join(path, "modality_categorical_covariate_keys.npy"), allow_pickle=True
)
covariate_lengths = np.load(os.path.join(path, "covariate_lengths.npy"), allow_pickle=True)
continuous_covariate_keys = np.load(
    os.path.join(path, "continuous_covariate_keys.npy"), allow_pickle=True
)


In [ ]:
covariate_names = np.concatenate([modality_categorical_covariate_keys, continuous_covariate_keys])
covariate_names


In [ ]:
with open(os.path.join(path, "covariate_attention_registry_group_mean_phenotypes.pkl"), "rb") as f:
    covariate_attention_registry_group_mean_phenotypes = pickle.load(f)


In [ ]:
covariate_attention_registry_group_phenotypes = covariate_attention_registry_group_mean_phenotypes


In [ ]:
with open(
    os.path.join(path_shuff, "covariate_attention_registry_group_mean_phenotypes_with_ci.pkl"), "rb"
) as f:
    covariate_attention_registry_group_mean_phenotypes_with_ci_shuff = pickle.load(f)


In [ ]:
covariate_attention_registry_group_phenotypes_with_ci_shuff = (
    covariate_attention_registry_group_mean_phenotypes_with_ci_shuff
)


In [ ]:
with open(
    os.path.join(path_alt, "covariate_attention_registry_group_mean_phenotypes_with_ci.pkl"), "rb"
) as f:
    covariate_attention_registry_group_mean_phenotypes_with_ci = pickle.load(f)


In [ ]:
covariate_attention_registry_group_phenotypes_with_ci = (
    covariate_attention_registry_group_mean_phenotypes_with_ci
)


In [ ]:
with open(
    os.path.join(path_alt, "covariate_attention_registry_group_ci_phenotypes_with_ci.pkl"), "rb"
) as f:
    covariate_attention_registry_group_ci_phenotypes_with_ci = pickle.load(f)


In [ ]:
covariate_attention_registry_group_phenotypes_ci_with_ci = (
    covariate_attention_registry_group_ci_phenotypes_with_ci
)


In [ ]:
import numpy as np

results = {}

for cell_type, modalities in covariate_attention_registry_group_phenotypes_with_ci.items():

    results[cell_type] = {}

    for modality, go_dict in modalities.items():

        values_dict = covariate_attention_registry_group_phenotypes_with_ci[cell_type][modality]
        ci_dict = covariate_attention_registry_group_phenotypes_ci_with_ci[cell_type][modality]

        diffs = []

        for go_term, values in values_dict.items():

            ci = ci_dict[go_term]

            lower = ci[:, 0]
            upper = ci[:, 1]

            diff_lower = np.abs(values - lower)
            diff_upper = np.abs(values - upper)

            diff = np.minimum(diff_lower, diff_upper)

            diffs.extend(diff)

        diffs = np.array(diffs)

        results[cell_type][modality] = {
            "mean_diff": diffs.mean(),
            "max_diff": diffs.max(),
            "min_diff": diffs.min(),
        }

print(results)


In [ ]:
if mode == "all":
    phenotypes = [
        "HSC",
        "Lymph_prog",
        "G_M_prog",
        "MK_E_prog",
        "ILC",
        "ILC1",
        "NK",
        "NK_CD158e1+",
        "CD4+_T_naive",
        "CD4+_T_activated",
        "CD4+_T_activated_integrinB7+",
        "CD4+_T_CD314+_CD45RA+",
        "CD8+_T_naive",
        "CD8+_T_naive_CD127+_CD26-_CD101-",
        "T_prog_cycling",
        "CD8+_T_CD49f+",
        "CD8+_T_CD69+_CD45RA+",
        "CD8+_T_CD69+_CD45RO+",
        "CD8+_T_TIGIT+_CD45RA+",
        "CD8+_T_TIGIT+_CD45RO+",
        "CD8+_T_CD57+_CD45RA+",
        "CD8+_T_CD57+_CD45RO+",
        "gdT_CD158b+",
        "gdT_TCRVD2+",
        "dnT",
        "T_reg",
        "Naive_CD20+_B_IGKC+",
        "Naive_CD20+_B_IGKC-",
        "Transitional_B",
        "B1_B_IGKC+",
        "B1_B_IGKC-",
        "Plasma_cell_IGKC+",
        "Plasma_cell_IGKC-",
        "Plasmablast_IGKC+",
        "Plasmablast_IGKC-",
        "cDC1",
        "cDC2",
        "pDC",
        "CD14+_Mono",
        "CD16+_Mono",
        "Erythroblast",
        "Normoblast",
        "Proerythroblast",
        "Reticulocyte",
        "MAIT",
    ]
    phenotypes_labels = [
        "HSC",
        "Lymph prog",
        "G/M prog",
        "MK/E prog",
        "ILC",
        "ILC1",
        "NK",
        "NK CD158e1+",
        "CD4+ T naive",
        "CD4+ T activated",
        "CD4+ T activated B7+",
        "CD4+ T CD314+ CD45RA+",
        "CD8+ T naive",
        "CD8+ T naive CD127+ CD26- CD101-",
        "T prog cycling",
        "CD8+ T CD49f+",
        "CD8+ T CD69+ CD45RA+",
        "CD8+ T CD69+ CD45RO+",
        "CD8+ T TIGIT+ CD45RA+",
        "CD8+ T TIGIT+ CD45RO+",
        "CD8+ T CD57+ CD45RA+",
        "CD8+ T CD57+ CD45RO+",
        "gdT CD158b+",
        "gdT TCRVD2+",
        "dnT",
        "T reg",
        "Naive CD20+ B IGKC+",
        "Naive CD20+ B IGKC-",
        "Transitional B",
        "B1 B IGKC+",
        "B1 B IGKC-",
        "Plasma cell IGKC+",
        "Plasma cell IGKC-",
        "Plasmablast IGKC+",
        "Plasmablast IGKC-",
        "cDC1",
        "cDC2",
        "pDC",
        "CD14+ Mono",
        "CD16+ Mono",
        "Erythroblast",
        "Normoblast",
        "Proerythroblast",
        "Reticulocyte",
        "MAIT",
    ]
    phenotypes_dataset = phenotypes_labels
else:
    phenotypes = [
        "Lymph_prog",
        "CD4+_T_naive",
        "CD4+_T_activated",
        "CD4+_T_activated_integrinB7+",
        "CD4+_T_CD314+_CD45RA+",
        "CD8+_T_naive",
        "CD8+_T_naive_CD127+_CD26-_CD101-",
        "CD8+_T_TIGIT+_CD45RA+",
        "CD8+_T_TIGIT+_CD45RO+",
        "CD8+_T_CD57+_CD45RA+",
        "CD8+_T_CD57+_CD45RO+",
        "Naive_CD20+_B_IGKC+",
        "Naive_CD20+_B_IGKC-",
        "Erythroblast",
        "Normoblast",
        "Reticulocyte",
    ]
    phenotypes_labels = [
        "Lymph prog",
        "CD4+ T naive",
        "CD4+ T activated",
        "CD4+ T activated integrinB7+",
        "CD4+ T CD314+ CD45RA+",
        "CD8+ T naive",
        "CD8+ T naive CD127+ CD26- CD101-",
        "CD8+ T TIGIT+ CD45RA+",
        "CD8+ T TIGIT+ CD45RO+",
        "CD8+ T CD57+ CD45RA+",
        "CD8+ T CD57+ CD45RO+",
        "Naive CD20+ B IGKC+",
        "Naive CD20+ B IGKC-",
        "Erythroblast",
        "Normoblast",
        "Reticulocyte",
    ]
    phenotypes_dataset = phenotypes_labels
    phenotypes = [
        "Lymph_prog",
        "G_M_prog",
        "MK_E_prog",
        "NK",
        "CD4+_T_naive",
        "CD4+_T_activated",
        "CD4+_T_activated_integrinB7+",
        "CD4+_T_CD314+_CD45RA+",
        "CD8+_T_naive",
        "CD8+_T_TIGIT+_CD45RA+",
        "CD8+_T_TIGIT+_CD45RO+",
        "CD8+_T_CD57+_CD45RA+",
        "CD8+_T_CD57+_CD45RO+",
        "Naive_CD20+_B_IGKC+",
        "Naive_CD20+_B_IGKC-",
        "Plasma_cell_IGKC+",
        "Plasma_cell_IGKC-",
        "Plasmablast_IGKC+",
        "Plasmablast_IGKC-",
        "cDC1",
        "cDC2",
        "CD14+_Mono",
        "CD16+_Mono",
        "Erythroblast",
        "Normoblast",
        "Reticulocyte",
    ]
    phenotypes_labels = [
        "Lymph prog",
        "G/M prog",
        "MK/E prog",
        "NK",
        "CD4+ T naive",
        "CD4+ T activated",
        "CD4+ T activated integrinB7+",
        "CD4+ T CD314+ CD45RA+",
        "CD8+ T naive",
        "CD8+ T TIGIT+ CD45RA+",
        "CD8+ T TIGIT+ CD45RO+",
        "CD8+ T CD57+ CD45RA+",
        "CD8+ T CD57+ CD45RO+",
        "Naive CD20+ B IGKC+",
        "Naive CD20+ B IGKC-",
        "Plasma cell IGKC+",
        "Plasma cell IGKC-",
        "Plasmablast IGKC+",
        "Plasmablast IGKC-",
        "cDC1",
        "cDC2",
        "CD14+ Mono",
        "CD16+ Mono",
        "Erythroblast",
        "Normoblast",
        "Reticulocyte",
    ]
    phenotypes_dataset = phenotypes_labels


In [ ]:
(
    mdata,
    mdata_counts,
    adata,
    train_indices,
    val_indices,
    n_genes,
    n_regions,
    n_snps,
    labels_key,
    further_labels_keys,
    batch_key,
    patient_key,
    further_batch_keys,
    further_continuous_batch_keys,
    n_patient_covariates,
    protein_expression_obsm_key,
) = datasets.load_neurips2021_cite_BMMC(1.0, False, True)


In [ ]:
gex_age_corr = [
    scipy.stats.spearmanr(
        mdata_counts.mod["rna"].obs["DonorAge"], mdata_counts.mod["rna"].X[:, gene_index]
    ).statistic
    for gene_index in tqdm(range(len(mdata_counts.mod["rna"].var)))
]
gex_age_corr


In [ ]:
genes_parent_go0002682 = [
    mdata_counts.mod["rna"].var[mdata_counts.mod["rna"].var["gene_stable_id"] == gene].index[0]
    for gene in set(result_go0002682)
    & (
        set(
            mdata_counts.mod["rna"].var.iloc[np.where(np.array(gex_age_corr) < -0.1)][
                "gene_stable_id"
            ]
        )
        | set(
            mdata_counts.mod["rna"].var.iloc[np.where(np.array(gex_age_corr) > 0.1)][
                "gene_stable_id"
            ]
        )
    )
]


In [ ]:
import scanpy as sc

mdata_counts_dotplot_subset = mdata_counts.mod["rna"][
    mdata_counts.mod["rna"].obs[labels_key].isin(phenotypes_labels_dotplots)
].copy()
if hasattr(mdata_counts_dotplot_subset.obs[labels_key], "cat"):
    mdata_counts_dotplot_subset.obs[labels_key] = mdata_counts_dotplot_subset.obs[labels_key].cat.remove_unused_categories()

_dp = sc.pl.dotplot(
    mdata_counts_dotplot_subset,
    ["CCL4", "CCL5", "CST7", "KLRB1", "KLRD1"],
    categories_order=phenotypes_labels_dotplots,
    groupby=labels_key,
    show=False,
    return_fig=True,
)
_dp.savefig(
    os.path.join(_OUTDIR_ABS, "neurips2021_cite_bmmc_age_correlated_genes_dotplot.png"),
    dpi=500, format="png", transparent=True, bbox_inches="tight",
)
_dp.savefig(
    os.path.join(_OUTDIR_ABS, "neurips2021_cite_bmmc_age_correlated_genes_dotplot.svg"),
    dpi=500, transparent=True, bbox_inches="tight", format="svg",
)
plt.show()


In [ ]:
plt.scatter(mdata_counts.mod["rna"].obs["DonorAge"], mdata_counts.mod["rna"].X[:, 145])
plt.show()


In [ ]:
np.min(gex_age_corr)


In [ ]:
np.where(np.array(gex_age_corr) > 0.1)


In [ ]:
len(np.unique(mdata_counts.mod["rna"].obs["DonorID"]))


In [ ]:
mdata_counts.mod["rna"].obs["DonorAge"] = mdata_counts.mod["rna"].obs["DonorAge"].astype("category")


In [ ]:
len(np.unique(mdata_counts.mod["rna"].obs["DonorAge"]))


In [ ]:
for gene in ["IFNG", "BCL6", "BACH1", "S1PR1", "S1PR3", "TNF"]:
    print(gex_age_corr[np.where(mdata_counts.mod["rna"].var.index == gene)[0][0]])


In [ ]:
from matplotlib.colors import TwoSlopeNorm

def plot_covariate_attention_heatmap(mode, go_ensembl_phenotypes, covariate_attention_registry_group_phenotypes_heatmap, covariate_attention_registry_group_phenotypes_indices, save_name, figsize=(7, 8)):
    heatmap_data = covariate_attention_registry_group_phenotypes_heatmap[:, covariate_attention_registry_group_phenotypes_indices]
    df = pd.DataFrame(heatmap_data, columns=[go_ensembl_phenotypes[i] for i in covariate_attention_registry_group_phenotypes_indices], index=phenotypes_labels)
    print(heatmap_data.shape)
    vmin = 0
    vmax = heatmap_data.max()
    center = np.mean(covariate_attention_registry_group_phenotypes_heatmap.flatten())
    norm = TwoSlopeNorm(vmin=vmin, vcenter=center, vmax=vmax)
    if mode == 'all':
        g = sns.clustermap(df, cmap=cmap_heat, norm=norm, figsize=figsize, xticklabels=True, yticklabels=True)
    else:
        g = sns.clustermap(df, cmap=cmap_heat2, norm=norm, figsize=figsize, xticklabels=True, yticklabels=True, cbar_kws={'orientation': 'horizontal'})
    if mode != 'all':
        print(mode)
        g.ax_col_dendrogram.set_visible(False)
        g.ax_row_dendrogram.set_visible(False)
    if mode == 'all':
        g.cax.set_position([1.05, 0.5, 0.03, 0.2])
    else:
        g.cax.set_position([0.15, 0.9, 0.4, 0.02])
        g.cax.xaxis.set_ticks_position('top')
        g.cax.xaxis.set_label_position('top')
        g.cax.yaxis.set_ticks([])
        g.cax.yaxis.set_visible(False)
    if mode != 'all':
        print('Actually running.')
        for label in g.ax_heatmap.get_yticklabels():
            label.set_rotation(180)
    if mode != 'all':
        plt.tight_layout()
    ax = g.ax_heatmap
    ax.grid(False)
    if save_name:
        plt.savefig(os.path.join(_OUTDIR_ABS, f'{save_name}.png'), dpi=500, format='png', transparent=True, bbox_inches='tight')
        plt.savefig(os.path.join(_OUTDIR_ABS, f'{save_name}.svg'), dpi=500, transparent=True, bbox_inches='tight', format='svg')
    else:
        plt.show()


In [ ]:
modality = "expression"


In [ ]:
go_ensembl_terms = np.array(
    list(covariate_attention_registry_group_mean_phenotypes[phenotypes[0]][modality].keys())
)
go_terms = np.array(
    [
        (
            f"{obodag[go_ensembl_term].name}"
            if go_ensembl_term in obodag.keys()
            else go_ensembl_term
        )
        for go_ensembl_term in go_ensembl_terms
        if not go_ensembl_term.startswith("ENSG")
    ]
)
ensembl_terms = np.array(
    [
        (
            ensembl_gene_id_external_gene_name[
                ensembl_gene_id_external_gene_name["gene_stable_id"] == go_ensembl_term
            ]["Gene name"].iloc[0]
            if go_ensembl_term in list(ensembl_gene_id_external_gene_name["gene_stable_id"])
            else go_ensembl_term
        )
        for go_ensembl_term in tqdm(go_ensembl_terms)
        if not go_ensembl_term.startswith("GO:")
    ]
)
go_ensembl_terms = np.concatenate([go_terms, ensembl_terms])


In [ ]:
covariate_index = 3
covariate_name = covariate_names[covariate_index]


In [ ]:
for go_ensembl_switch_name, go_ensembl_switch, go_ensembl_phenotypes in zip(
    ["go", "ensembl"], ["GO:", "ENSG"], [go_terms, ensembl_terms]
):
    break


In [ ]:
covariate_attention_registry_group_phenotypes_heatmap = np.array(
    [
        [
            covariate_attention_registry_group_phenotypes[phenotype][modality][go_ensembl][
                covariate_index
            ]
            for go_ensembl in covariate_attention_registry_group_phenotypes[phenotype][
                modality
            ].keys()
            if go_ensembl.startswith(go_ensembl_switch)
        ]
        for phenotype in phenotypes
    ]
)
if (
    np.max(covariate_attention_registry_group_phenotypes_heatmap)
    / np.mean(covariate_attention_registry_group_phenotypes_heatmap)
    > 1.001
    and np.mean(covariate_attention_registry_group_phenotypes_heatmap)
    / np.min(covariate_attention_registry_group_phenotypes_heatmap)
    > 1.001
):
    print(f"{modality} {covariate_name} {go_ensembl_switch_name}")
    print(covariate_attention_registry_group_phenotypes_heatmap.shape)
    print(np.mean(covariate_attention_registry_group_phenotypes_heatmap))
    print(np.max(covariate_attention_registry_group_phenotypes_heatmap))
    print(np.min(covariate_attention_registry_group_phenotypes_heatmap))
    if go_ensembl_switch_name == "go":
        print("GO:UNKNOWN")
        print(
            np.mean(
                [
                    covariate_attention_registry_group_phenotypes[phenotype][modality][
                        "GO:UNKNOWN1"
                    ][covariate_index]
                    for phenotype in phenotypes
                ]
            )
        )
        print(
            np.std(
                [
                    covariate_attention_registry_group_phenotypes[phenotype][modality][
                        "GO:UNKNOWN1"
                    ][covariate_index]
                    for phenotype in phenotypes
                ]
            )
        )
        print(
            np.mean(
                [
                    covariate_attention_registry_group_phenotypes[phenotype][modality][
                        "GO:UNKNOWN2"
                    ][covariate_index]
                    for phenotype in phenotypes
                ]
            )
        )
        print(
            np.std(
                [
                    covariate_attention_registry_group_phenotypes[phenotype][modality][
                        "GO:UNKNOWN2"
                    ][covariate_index]
                    for phenotype in phenotypes
                ]
            )
        )
        print(
            np.mean(
                [
                    covariate_attention_registry_group_phenotypes[phenotype][modality][
                        "GO:UNKNOWN3"
                    ][covariate_index]
                    for phenotype in phenotypes
                ]
            )
        )
        print(
            np.std(
                [
                    covariate_attention_registry_group_phenotypes[phenotype][modality][
                        "GO:UNKNOWN3"
                    ][covariate_index]
                    for phenotype in phenotypes
                ]
            )
        )

    if go_ensembl_switch_name == "go":
        if modality == "protein":
            lower_selection_criterium = 0.05
        else:
            lower_selection_criterium = 0.005
    else:
        lower_selection_criterium = 0.0001
    upper_selection_criterium = 1 - lower_selection_criterium
    covariate_attention_registry_group_phenotypes_indices = np.unique(
        np.where(
            (
                covariate_attention_registry_group_phenotypes_heatmap
                <= np.quantile(
                    covariate_attention_registry_group_phenotypes_heatmap.flatten(),
                    lower_selection_criterium,
                )
            )
            | (
                covariate_attention_registry_group_phenotypes_heatmap
                >= np.quantile(
                    covariate_attention_registry_group_phenotypes_heatmap.flatten(),
                    upper_selection_criterium,
                )
            )
        )[1]
    )

    print("covariate_attention_registry_group_phenotypes_indices shape")
    print(covariate_attention_registry_group_phenotypes_indices.shape)
    print(go_ensembl_phenotypes.shape)
    print(covariate_attention_registry_group_phenotypes_heatmap.shape)

    if modality == "expression" and covariate_name == "DonorAge" and go_ensembl_switch_name == "go":
        if mode == "subset":
            go_ensembl_phenotypes_restrict = [
                "regulation of metabolic process",
                "regulation of signaling",
                "regulation of immune system process",
                "protein metabolic process",
                "cell development",
                "cellular process",
                "platelet activation",
            ]
            covariate_attention_registry_group_phenotypes_indices = [
                list(go_ensembl_phenotypes).index(go_ensembl_phenotype)
                for go_ensembl_phenotype in go_ensembl_phenotypes_restrict
            ]
            figsize = (1.82, 4.5)
        else:
            figsize = (7, 8)
        plot_covariate_attention_heatmap(
            mode,
            go_ensembl_phenotypes,
            covariate_attention_registry_group_phenotypes_heatmap,
            covariate_attention_registry_group_phenotypes_indices,
            f"neurips2021_cite_bmmc_covariate_attention_{mode}_{modality}_{covariate_name}_{go_ensembl_switch_name}",
            figsize=figsize,
        )

    try:
        plot_covariate_attention_heatmap(
            mode,
            go_ensembl_phenotypes,
            covariate_attention_registry_group_phenotypes_heatmap,
            covariate_attention_registry_group_phenotypes_indices,
            f"neurips2021_cite_bmmc_covariate_attention_{mode}_{modality}_{covariate_name}_{go_ensembl_switch_name}",
            figsize=figsize,
        )
    except:
        print(
            f"not run {modality} {covariate_name} {go_ensembl_switch_name} due to low variability ||"
        )
else:
    print(f"not run {modality} {covariate_name} {go_ensembl_switch_name} due to low variability |")


In [ ]:
covariate_attention_registry_group_phenotypes_with_ci_shuff_heatmap = np.array(
    [
        [
            covariate_attention_registry_group_phenotypes_with_ci_shuff[phenotype][modality][
                go_ensembl
            ][covariate_index]
            for go_ensembl in covariate_attention_registry_group_phenotypes_with_ci_shuff[
                phenotype
            ][modality].keys()
            if go_ensembl.startswith(go_ensembl_switch)
        ]
        for phenotype in phenotypes
    ]
)
if (
    np.max(covariate_attention_registry_group_phenotypes_with_ci_shuff_heatmap)
    / np.mean(covariate_attention_registry_group_phenotypes_with_ci_shuff_heatmap)
    > 1.001
    and np.mean(covariate_attention_registry_group_phenotypes_with_ci_shuff_heatmap)
    / np.min(covariate_attention_registry_group_phenotypes_with_ci_shuff_heatmap)
    > 1.001
):
    print(f"{modality} {covariate_name} {go_ensembl_switch_name}")
    print(covariate_attention_registry_group_phenotypes_with_ci_shuff_heatmap.shape)
    print(np.mean(covariate_attention_registry_group_phenotypes_with_ci_shuff_heatmap))
    print(np.max(covariate_attention_registry_group_phenotypes_with_ci_shuff_heatmap))
    print(np.min(covariate_attention_registry_group_phenotypes_with_ci_shuff_heatmap))
    if go_ensembl_switch_name == "go":
        print("GO:UNKNOWN")
        print(
            np.mean(
                [
                    covariate_attention_registry_group_phenotypes_with_ci_shuff[phenotype][
                        modality
                    ]["GO:UNKNOWN1"][covariate_index]
                    for phenotype in phenotypes
                ]
            )
        )
        print(
            np.std(
                [
                    covariate_attention_registry_group_phenotypes_with_ci_shuff[phenotype][
                        modality
                    ]["GO:UNKNOWN1"][covariate_index]
                    for phenotype in phenotypes
                ]
            )
        )
        print(
            np.mean(
                [
                    covariate_attention_registry_group_phenotypes_with_ci_shuff[phenotype][
                        modality
                    ]["GO:UNKNOWN2"][covariate_index]
                    for phenotype in phenotypes
                ]
            )
        )
        print(
            np.std(
                [
                    covariate_attention_registry_group_phenotypes_with_ci_shuff[phenotype][
                        modality
                    ]["GO:UNKNOWN2"][covariate_index]
                    for phenotype in phenotypes
                ]
            )
        )
        print(
            np.mean(
                [
                    covariate_attention_registry_group_phenotypes_with_ci_shuff[phenotype][
                        modality
                    ]["GO:UNKNOWN3"][covariate_index]
                    for phenotype in phenotypes
                ]
            )
        )
        print(
            np.std(
                [
                    covariate_attention_registry_group_phenotypes_with_ci_shuff[phenotype][
                        modality
                    ]["GO:UNKNOWN3"][covariate_index]
                    for phenotype in phenotypes
                ]
            )
        )

    if go_ensembl_switch_name == "go":
        if modality == "protein":
            lower_selection_criterium = 0.05
        else:
            lower_selection_criterium = 0.005
    else:
        lower_selection_criterium = 0.0001
    upper_selection_criterium = 1 - lower_selection_criterium
    covariate_attention_registry_group_phenotypes_with_ci_shuff_indices = np.unique(
        np.where(
            (
                covariate_attention_registry_group_phenotypes_with_ci_shuff_heatmap
                <= np.quantile(
                    covariate_attention_registry_group_phenotypes_with_ci_shuff_heatmap.flatten(),
                    lower_selection_criterium,
                )
            )
            | (
                covariate_attention_registry_group_phenotypes_with_ci_shuff_heatmap
                >= np.quantile(
                    covariate_attention_registry_group_phenotypes_with_ci_shuff_heatmap.flatten(),
                    upper_selection_criterium,
                )
            )
        )[1]
    )

    print("covariate_attention_registry_group_phenotypes_with_ci_shuff_indices shape")
    print(covariate_attention_registry_group_phenotypes_with_ci_shuff_indices.shape)
    print(go_ensembl_phenotypes.shape)
    print(covariate_attention_registry_group_phenotypes_with_ci_shuff_heatmap.shape)

    if modality == "expression" and covariate_name == "DonorAge" and go_ensembl_switch_name == "go":
        if mode == "subset":
            go_ensembl_phenotypes_restrict = [
                "regulation of metabolic process",
                "regulation of signaling",
                "regulation of immune system process",
                "protein metabolic process",
                "cell development",
                "cellular process",
                "platelet activation",
            ]
            covariate_attention_registry_group_phenotypes_with_ci_shuff_indices = [
                list(go_ensembl_phenotypes).index(go_ensembl_phenotype)
                for go_ensembl_phenotype in go_ensembl_phenotypes_restrict
            ]
            figsize = (1.82, 4.5)
        else:
            figsize = (7, 8)
        plot_covariate_attention_heatmap(
            mode,
            go_ensembl_phenotypes,
            covariate_attention_registry_group_phenotypes_with_ci_shuff_heatmap,
            covariate_attention_registry_group_phenotypes_with_ci_shuff_indices,
            f"neurips2021_cite_bmmc_covariate_attention_shuff_{mode}_{modality}_{covariate_name}_{go_ensembl_switch_name}",
            figsize=figsize,
        )

else:
    print(f"not run {modality} {covariate_name} {go_ensembl_switch_name} due to low variability |")


In [ ]:
def _run_covariate_attention_pass():
    for modality in ["expression", "protein"]:

        go_ensembl_terms = np.array(
            list(covariate_attention_registry_group_mean_phenotypes[phenotypes[0]][modality].keys())
        )

        go_terms = np.array(
            [
                (
                    f"{obodag[go_ensembl_term].name}"
                    if go_ensembl_term in obodag.keys()
                    else go_ensembl_term
                )
                for go_ensembl_term in go_ensembl_terms
                if not go_ensembl_term.startswith("ENSG")
            ]
        )

        ensembl_terms = np.array(
            [
                (
                    ensembl_gene_id_external_gene_name[
                        ensembl_gene_id_external_gene_name["gene_stable_id"] == go_ensembl_term
                    ]["Gene name"].iloc[0]
                    if go_ensembl_term in list(ensembl_gene_id_external_gene_name["gene_stable_id"])
                    else go_ensembl_term
                )
                for go_ensembl_term in tqdm(go_ensembl_terms)
                if not go_ensembl_term.startswith("GO:")
            ]
        )

        go_ensembl_terms = np.concatenate([go_terms, ensembl_terms])

        for covariate_index in range(len(covariate_names)):

            covariate_name = covariate_names[covariate_index]

            for go_ensembl_switch_name, go_ensembl_switch, go_ensembl_phenotypes in zip(
                ["go", "ensembl"],
                ["GO:", "ENSG"],
                [go_terms, ensembl_terms],
            ):

                if modality == "protein" and go_ensembl_switch_name == "ensembl":
                    continue

                covariate_attention_registry_group_phenotypes_heatmap = np.array(
                    [
                        [
                            covariate_attention_registry_group_phenotypes[phenotype][modality][
                                go_ensembl
                            ][covariate_index]
                            for go_ensembl in covariate_attention_registry_group_phenotypes[phenotype][
                                modality
                            ].keys()
                            if go_ensembl.startswith(go_ensembl_switch)
                        ]
                        for phenotype in phenotypes
                    ]
                )

                if (
                    np.max(covariate_attention_registry_group_phenotypes_heatmap)
                    / np.mean(covariate_attention_registry_group_phenotypes_heatmap)
                    > 1.001
                    and np.mean(covariate_attention_registry_group_phenotypes_heatmap)
                    / np.min(covariate_attention_registry_group_phenotypes_heatmap)
                    > 1.001
                ):

                    print(f"{modality} {covariate_name} {go_ensembl_switch_name}")
                    print(covariate_attention_registry_group_phenotypes_heatmap.shape)
                    print(np.mean(covariate_attention_registry_group_phenotypes_heatmap))
                    print(np.max(covariate_attention_registry_group_phenotypes_heatmap))
                    print(np.min(covariate_attention_registry_group_phenotypes_heatmap))

                    if go_ensembl_switch_name == "go":
                        print("GO:UNKNOWN")
                        print(
                            np.mean(
                                [
                                    covariate_attention_registry_group_phenotypes[phenotype][modality][
                                        "GO:UNKNOWN1"
                                    ][covariate_index]
                                    for phenotype in phenotypes
                                ]
                            )
                        )
                        print(
                            np.std(
                                [
                                    covariate_attention_registry_group_phenotypes[phenotype][modality][
                                        "GO:UNKNOWN1"
                                    ][covariate_index]
                                    for phenotype in phenotypes
                                ]
                            )
                        )
                        print(
                            np.mean(
                                [
                                    covariate_attention_registry_group_phenotypes[phenotype][modality][
                                        "GO:UNKNOWN2"
                                    ][covariate_index]
                                    for phenotype in phenotypes
                                ]
                            )
                        )
                        print(
                            np.std(
                                [
                                    covariate_attention_registry_group_phenotypes[phenotype][modality][
                                        "GO:UNKNOWN2"
                                    ][covariate_index]
                                    for phenotype in phenotypes
                                ]
                            )
                        )
                        print(
                            np.mean(
                                [
                                    covariate_attention_registry_group_phenotypes[phenotype][modality][
                                        "GO:UNKNOWN3"
                                    ][covariate_index]
                                    for phenotype in phenotypes
                                ]
                            )
                        )
                        print(
                            np.std(
                                [
                                    covariate_attention_registry_group_phenotypes[phenotype][modality][
                                        "GO:UNKNOWN3"
                                    ][covariate_index]
                                    for phenotype in phenotypes
                                ]
                            )
                        )

                    if go_ensembl_switch_name == "go":
                        if modality == "protein":
                            lower_selection_criterium = 0.05
                        else:
                            lower_selection_criterium = 0.005
                    else:
                        lower_selection_criterium = 0.0001

                    upper_selection_criterium = 1 - lower_selection_criterium

                    covariate_attention_registry_group_phenotypes_indices = np.unique(
                        np.where(
                            (
                                covariate_attention_registry_group_phenotypes_heatmap
                                <= np.quantile(
                                    covariate_attention_registry_group_phenotypes_heatmap.flatten(),
                                    lower_selection_criterium,
                                )
                            )
                            | (
                                covariate_attention_registry_group_phenotypes_heatmap
                                >= np.quantile(
                                    covariate_attention_registry_group_phenotypes_heatmap.flatten(),
                                    upper_selection_criterium,
                                )
                            )
                        )[1]
                    )

                    print("covariate_attention_registry_group_phenotypes_indices shape")
                    print(covariate_attention_registry_group_phenotypes_indices.shape)
                    print(go_ensembl_phenotypes.shape)
                    print(covariate_attention_registry_group_phenotypes_heatmap.shape)

                    ci_lower_heatmap = np.array(
                        [
                            [
                                covariate_attention_registry_group_phenotypes_ci_with_ci[phenotype][
                                    modality
                                ][go_ensembl][covariate_index][0]
                                for go_ensembl in covariate_attention_registry_group_phenotypes_ci_with_ci[
                                    phenotype
                                ][
                                    modality
                                ].keys()
                                if go_ensembl.startswith(go_ensembl_switch)
                            ]
                            for phenotype in phenotypes
                        ]
                    )

                    ci_upper_heatmap = np.array(
                        [
                            [
                                covariate_attention_registry_group_phenotypes_ci_with_ci[phenotype][
                                    modality
                                ][go_ensembl][covariate_index][1]
                                for go_ensembl in covariate_attention_registry_group_phenotypes_ci_with_ci[
                                    phenotype
                                ][
                                    modality
                                ].keys()
                                if go_ensembl.startswith(go_ensembl_switch)
                            ]
                            for phenotype in phenotypes
                        ]
                    )

                    vals_subset = covariate_attention_registry_group_phenotypes_heatmap[
                        :, covariate_attention_registry_group_phenotypes_indices
                    ]

                    lower_subset = ci_lower_heatmap[
                        :, covariate_attention_registry_group_phenotypes_indices
                    ]

                    upper_subset = ci_upper_heatmap[
                        :, covariate_attention_registry_group_phenotypes_indices
                    ]

                    diff_lower = np.abs(vals_subset - lower_subset)
                    diff_upper = np.abs(vals_subset - upper_subset)

                    diff = np.minimum(diff_lower, diff_upper)

                    diff_flat = diff.flatten()

                    print("CI deviation summary")
                    print(f"ROUND: {mode} | {modality} | {covariate_name} | {go_ensembl_switch_name}")
                    print("subset size:", diff_flat.shape[0])
                    print("mean diff:", diff_flat.mean())
                    print("max diff :", diff_flat.max())
                    print("min diff :", diff_flat.min())

                    if (
                        modality == "expression"
                        and covariate_name == "DonorAge"
                        and go_ensembl_switch_name == "go"
                    ):

                        if mode == "subset":
                            go_ensembl_phenotypes_restrict = [
                                "regulation of metabolic process",
                                "regulation of signaling",
                                "regulation of immune system process",
                                "protein metabolic process",
                                "cell development",
                                "cellular process",
                                "platelet activation",
                            ]

                            covariate_attention_registry_group_phenotypes_indices = [
                                list(go_ensembl_phenotypes).index(go_ensembl_phenotype)
                                for go_ensembl_phenotype in go_ensembl_phenotypes_restrict
                            ]

                            figsize = (1.82, 4.5)

                        else:
                            figsize = (7, 8)

                        plot_covariate_attention_heatmap(
                            mode,
                            go_ensembl_phenotypes,
                            covariate_attention_registry_group_phenotypes_heatmap,
                            covariate_attention_registry_group_phenotypes_indices,
                            f"neurips2021_cite_bmmc_covariate_attention_{mode}_{modality}_{covariate_name}_{go_ensembl_switch_name}",
                            figsize=figsize,
                        )

                        covariate_attention_registry_group_phenotypes_with_ci_shuff_heatmap = np.array(
                            [
                                [
                                    covariate_attention_registry_group_phenotypes_with_ci_shuff[
                                        phenotype
                                    ][modality][go_ensembl][covariate_index]
                                    for go_ensembl in covariate_attention_registry_group_phenotypes_with_ci_shuff[
                                        phenotype
                                    ][
                                        modality
                                    ].keys()
                                    if go_ensembl.startswith(go_ensembl_switch)
                                ]
                                for phenotype in phenotypes
                            ]
                        )

                        plot_covariate_attention_heatmap(
                            mode,
                            go_ensembl_phenotypes,
                            covariate_attention_registry_group_phenotypes_with_ci_shuff_heatmap,
                            covariate_attention_registry_group_phenotypes_indices,
                            f"neurips2021_cite_bmmc_covariate_attention_shuff_{mode}_{modality}_{covariate_name}_{go_ensembl_switch_name}",
                            figsize=figsize,
                        )

                else:

                    print(
                        f"not run {modality} {covariate_name} {go_ensembl_switch_name} due to low variability |"
                    )

_run_covariate_attention_pass()


In [ ]:
mode = "all"

phenotypes = [
    "HSC", "Lymph_prog", "G_M_prog", "MK_E_prog", "ILC", "ILC1", "NK", "NK_CD158e1+",
    "CD4+_T_naive", "CD4+_T_activated", "CD4+_T_activated_integrinB7+",
    "CD4+_T_CD314+_CD45RA+", "CD8+_T_naive", "CD8+_T_naive_CD127+_CD26-_CD101-",
    "T_prog_cycling", "CD8+_T_CD49f+", "CD8+_T_CD69+_CD45RA+", "CD8+_T_CD69+_CD45RO+",
    "CD8+_T_TIGIT+_CD45RA+", "CD8+_T_TIGIT+_CD45RO+", "CD8+_T_CD57+_CD45RA+",
    "CD8+_T_CD57+_CD45RO+", "gdT_CD158b+", "gdT_TCRVD2+", "dnT", "T_reg",
    "Naive_CD20+_B_IGKC+", "Naive_CD20+_B_IGKC-", "Transitional_B", "B1_B_IGKC+",
    "B1_B_IGKC-", "Plasma_cell_IGKC+", "Plasma_cell_IGKC-", "Plasmablast_IGKC+",
    "Plasmablast_IGKC-", "cDC1", "cDC2", "pDC", "CD14+_Mono", "CD16+_Mono",
    "Erythroblast", "Normoblast", "Proerythroblast", "Reticulocyte", "MAIT",
]
phenotypes_labels = [
    "HSC", "Lymph prog", "G/M prog", "MK/E prog", "ILC", "ILC1", "NK", "NK CD158e1+",
    "CD4+ T naive", "CD4+ T activated", "CD4+ T activated B7+",
    "CD4+ T CD314+ CD45RA+", "CD8+ T naive", "CD8+ T naive CD127+ CD26- CD101-",
    "T prog cycling", "CD8+ T CD49f+", "CD8+ T CD69+ CD45RA+", "CD8+ T CD69+ CD45RO+",
    "CD8+ T TIGIT+ CD45RA+", "CD8+ T TIGIT+ CD45RO+", "CD8+ T CD57+ CD45RA+",
    "CD8+ T CD57+ CD45RO+", "gdT CD158b+", "gdT TCRVD2+", "dnT", "T reg",
    "Naive CD20+ B IGKC+", "Naive CD20+ B IGKC-", "Transitional B", "B1 B IGKC+",
    "B1 B IGKC-", "Plasma cell IGKC+", "Plasma cell IGKC-", "Plasmablast IGKC+",
    "Plasmablast IGKC-", "cDC1", "cDC2", "pDC", "CD14+ Mono", "CD16+ Mono",
    "Erythroblast", "Normoblast", "Proerythroblast", "Reticulocyte", "MAIT",
]
phenotypes_dataset = phenotypes_labels

_run_covariate_attention_pass()
